### Features and Classification Model (3-Hour Bins)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available, skipping XGBClassifier")

In [ ]:
df = pd.read_csv('../Project_datasets/hourly_grouped_rides.csv')
df['time_bin'] = pd.to_datetime(df['time_bin'])
df = df.sort_values('time_bin').reset_index(drop=True)
print(f"Shape: {df.shape}")
print(f"Date range: {df['time_bin'].min()} to {df['time_bin'].max()}")
print(f"Zero bins: {(df['trip_count'] == 0).sum()} ({(df['trip_count'] == 0).mean():.1%})")
df.head()

#### Calendar and Time Features

In [ ]:
df['bin_hour'] = df['time_bin'].dt.hour  # start hour of the 3h bin (0, 3, 6, 9, 12, 15, 18, 21)
df['day_of_week'] = df['time_bin'].dt.weekday
df['day_of_month'] = df['time_bin'].dt.day
df['month'] = df['time_bin'].dt.month
df['year'] = df['time_bin'].dt.year
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Season
def get_season(month):
    if month in [12, 1, 2]:
        return 1  # Winter
    elif month in [3, 4, 5]:
        return 2  # Spring
    elif month in [6, 7, 8]:
        return 3  # Summer
    else:
        return 4  # Fall

df['season'] = df['month'].apply(get_season)

#### Time Period Features

In [ ]:
# Morning rush bin (6-9), evening rush bin (15-18)
df['is_morning_rush'] = (df['bin_hour'] == 6).astype(int)
df['is_evening_rush'] = (df['bin_hour'] == 15).astype(int)
df['is_rush'] = (df['is_morning_rush'] | df['is_evening_rush']).astype(int)

#### Cyclical Encoding

In [ ]:
# Bin of day (period = 8 bins per day)
df['bin_sin'] = np.sin(2 * np.pi * df['bin_hour'] / 24)
df['bin_cos'] = np.cos(2 * np.pi * df['bin_hour'] / 24)

# Day of week (period = 7)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Month (period = 12)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

#### Lag Features

In [ ]:
# With 3h bins: 1 step = 3h, 8 steps = 1 day, 56 steps = 1 week
df['lag_1'] = df['trip_count'].shift(1)       # previous 3h bin
df['lag_2'] = df['trip_count'].shift(2)       # 6h ago
df['lag_3'] = df['trip_count'].shift(3)       # 9h ago
df['lag_8'] = df['trip_count'].shift(8)       # same bin yesterday
df['lag_56'] = df['trip_count'].shift(56)     # same bin last week

#### Rolling Window Features

In [ ]:
# Rolling windows in 3h-bin steps
df['rolling_mean_3'] = df['trip_count'].shift(1).rolling(window=3).mean()    # ~9h window
df['rolling_mean_4'] = df['trip_count'].shift(1).rolling(window=4).mean()    # ~12h window
df['rolling_mean_8'] = df['trip_count'].shift(1).rolling(window=8).mean()    # ~24h window
df['rolling_std_8'] = df['trip_count'].shift(1).rolling(window=8).std()      # daily std
df['rolling_max_8'] = df['trip_count'].shift(1).rolling(window=8).max()      # daily max

# Same bin average over past 7 days
df['rolling_mean_same_bin_7d'] = df['trip_count'].shift(8).rolling(window=7, min_periods=1).mean()

#### Context Features

In [ ]:
# Total trips in previous day (8 bins)
df['trips_prev_day'] = df['trip_count'].shift(1).rolling(window=8).sum()

# Member ratio (avoid division by zero)
df['member_ratio'] = np.where(
    df['trip_count'] > 0,
    df['member_count'] / df['trip_count'],
    0
)

#### Interaction Features

In [ ]:
df['bin_x_weekend'] = df['bin_hour'] * df['is_weekend']
df['rush_x_weekday'] = df['is_rush'] * (1 - df['is_weekend'])

# Time period one-hot based on 3h bin start hour
def get_time_period(h):
    if h == 6:
        return 'morning'
    elif h in [9, 12]:
        return 'midday'
    elif h in [15, 18]:
        return 'evening'
    elif h == 21:
        return 'night'
    else:  # 0, 3
        return 'late_night'

df['time_period'] = df['bin_hour'].apply(get_time_period)
time_dummies = pd.get_dummies(df['time_period'], prefix='period')
df = pd.concat([df, time_dummies], axis=1)
df.drop(columns=['time_period'], inplace=True)

#### Handle Season Gaps and NaN Lags

In [ ]:
# Drop first week of each season (56 bins = 7 days) where lag features are NaN
season_starts = []
for year in range(2014, 2018):
    start = pd.Timestamp(f'{year}-04-01')
    end = start + pd.Timedelta(hours=56 * 3 - 1)  # first 56 bins (7 days)
    season_starts.append((start, end))

mask_season_start = pd.Series(False, index=df.index)
for start, end in season_starts:
    mask_season_start |= (df['time_bin'] >= start) & (df['time_bin'] <= end)

print(f"Rows before dropping season starts: {len(df)}")
df_model = df[~mask_season_start].copy()
print(f"Rows after dropping season starts: {len(df_model)}")
print(f"Dropped: {mask_season_start.sum()} rows")

#### Trip Count Binning (Classification Target)

In [ ]:
# Bin trip counts into 4 classes
df_model['trip_bin'] = pd.cut(
    df_model['trip_count'],
    bins=[-1, 0, 1, 3, np.inf],
    labels=['zero', 'low', 'mid', 'high']
)

print("Class distribution:")
print(df_model['trip_bin'].value_counts().sort_index())
print(f"\nClass percentages:")
print((df_model['trip_bin'].value_counts(normalize=True).sort_index() * 100).round(1))

# Encode labels for models
le = LabelEncoder()
le.fit(['zero', 'low', 'mid', 'high'])
df_model['trip_bin_encoded'] = le.transform(df_model['trip_bin'])

In [ ]:
df.to_csv('../Project_datasets/hourly_features.csv', index=False)
print(f"Saved hourly_features.csv ({len(df)} rows, {len(df.columns)} columns)")
print(f"\nFeature columns ({len(df.columns)}):")
print(list(df.columns))

#### Model Training

In [ ]:
# Define feature columns (exclude non-feature columns)
exclude_cols = ['time_bin', 'trip_count', 'avg_duration', 'member_count', 'trip_bin', 'trip_bin_encoded']
feature_cols = [c for c in df_model.columns if c not in exclude_cols]

print(f"Feature columns ({len(feature_cols)}):")
print(feature_cols)

# Time-based split
train = df_model[df_model['year'] < 2017]
test = df_model[df_model['year'] == 2017]

# Drop rows with NaN in features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)

X_train = train_clean[feature_cols]
y_train = train_clean['trip_bin_encoded']
X_test = test_clean[feature_cols]
y_test = test_clean['trip_bin_encoded']

# Apply SMOTE to balance training classes
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nTrain: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Train after SMOTE: {X_train_bal.shape[0]} rows")
print(f"\nSMOTE class distribution:")
print(pd.Series(y_train_bal).map(dict(enumerate(le.classes_))).value_counts().sort_index())

In [ ]:
models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
}

if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbosity=0
    )

results = []
predictions = {}
class_names = le.classes_

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_bal, y_train_bal)
    
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'model': name,
        'Accuracy': round(acc, 4),
        'F1_weighted': round(f1, 4)
    })
    predictions[name] = y_pred
    
    print(f"  Accuracy = {acc:.4f}, F1 (weighted) = {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names))

results_df = pd.DataFrame(results)
print("=" * 50)
print(results_df.to_string(index=False))

#### Cross-Validation (TimeSeriesSplit)

In [ ]:
# Full dataset for CV (drop NaN rows)
df_cv = df_model.dropna(subset=feature_cols)
X_all = df_cv[feature_cols]
y_all = df_cv['trip_bin_encoded']

tscv = TimeSeriesSplit(n_splits=5)

# Use best tree model for CV
cv_model_name = 'XGBoost' if HAS_XGB else 'GradientBoosting'
cv_model_class = type(models[cv_model_name])
cv_params = models[cv_model_name].get_params()

cv_scores = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_tr, y_val = y_all.iloc[train_idx], y_all.iloc[val_idx]
    
    # Apply SMOTE to each fold's training set
    smote_cv = SMOTE(random_state=42)
    X_tr_bal, y_tr_bal = smote_cv.fit_resample(X_tr, y_tr)
    
    cv_model = cv_model_class(**cv_params)
    cv_model.fit(X_tr_bal, y_tr_bal)
    y_pred_cv = cv_model.predict(X_val)
    
    fold_acc = accuracy_score(y_val, y_pred_cv)
    fold_f1 = f1_score(y_val, y_pred_cv, average='weighted')
    cv_scores.append({'fold': fold + 1, 'Accuracy': fold_acc, 'F1_weighted': fold_f1})
    print(f"Fold {fold+1}: Accuracy = {fold_acc:.4f}, F1 (weighted) = {fold_f1:.4f}")

cv_df = pd.DataFrame(cv_scores)
print(f"\nMean CV Accuracy: {cv_df['Accuracy'].mean():.4f} +/- {cv_df['Accuracy'].std():.4f}")
print(f"Mean CV F1 (weighted): {cv_df['F1_weighted'].mean():.4f} +/- {cv_df['F1_weighted'].std():.4f}")

#### Feature Importance

In [ ]:
# Use the best model for feature importance
best_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'model']
best_model = models[best_model_name]

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Plot top 20 features
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = min(20, len(feat_imp))
    feat_imp.head(top_n).plot.barh(x='feature', y='importance', ax=ax, legend=False)
    ax.set_xlabel('Importance')
    ax.set_title(f'Top {top_n} Feature Importances ({best_model_name})')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 features ({best_model_name}):")
    print(feat_imp.head(10).to_string(index=False))

#### Confusion Matrices

In [ ]:
n_models = len(predictions)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAccuracy = {acc:.4f}')

plt.tight_layout()
plt.show()

In [ ]:
results_df.to_csv('../Project_datasets/model_results_hourly_classification.csv', index=False)
print(f"Saved model_results_hourly_classification.csv")
print(f"\nFinal classification results:")
print(results_df.to_string(index=False))